# ARPD — Full Pipeline: LIAR + FEVER + Wikipedia
### Adaptive Retrieval with Paraphrase-robust training for fake news Detection

Notebook đầy đủ, gộp toàn bộ quy trình đã chạy thành công:

| Section | Nội dung |
|---|---|
| 0 | Setup: Drive, src/ files, libraries, LIAR dataset |
| 1 | Baselines: TF-IDF+LR, TF-IDF+SVM, MiniLM+MLP |
| 2 | ARPD (ImprovedMLP + Ensemble) trên LIAR |
| 3 | Statistical Significance — 5 seeds |
| 4 | Robustness Evaluation — paraphrase attack |
| 5 | Ablation Study |
| 6 | FEVER + Wikipedia: download, build wiki_index, extract evidence |
| 7 | ARPD trên FEVER (cross-domain) |
| 8 | Export kết quả cuối cùng |

> **Trước khi chạy**: chuẩn bị sẵn 8 file `.py` trong thư mục `src/` (data_loader.py, uncertainty_scorer.py, adaptive_retriever.py, paraphrase_augmentor.py, encoder.py, classifier.py, pipeline.py, __init__.py) và file `liar_dataset.zip` trên máy local để upload khi được yêu cầu.

> **Thời gian chạy toàn bộ**: ~90-120 phút (chủ yếu do download wiki-pages.zip 1.7GB và build index)

---
## Section 0 — Setup & Load LIAR

In [ ]:
# Cell 0.1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
# Cell 0.2 — Upload src/ files (8 file .py)
from google.colab import files
import os, shutil

os.makedirs('/content/src', exist_ok=True)
uploaded = files.upload()  # Chọn TẤT CẢ 8 file .py trong thư mục src/

for fname in uploaded.keys():
    shutil.copy(fname, f'/content/src/{fname}')
    shutil.copy(fname, f'/content/{fname}')
    print(f'✅ {fname}')

# Fix relative imports trong pipeline.py
for target in ['/content/src/pipeline.py', '/content/pipeline.py']:
    if os.path.exists(target):
        with open(target, 'r') as f:
            content = f.read()
        for mod in ['uncertainty_scorer', 'adaptive_retriever',
                    'paraphrase_augmentor', 'encoder', 'classifier', 'data_loader']:
            content = content.replace(f'from .{mod}', f'from {mod}')
        with open(target, 'w') as f:
            f.write(content)

print('\n✅ All src files ready!')

In [ ]:
# Cell 0.3 — Install dependencies
!pip install sentence-transformers transformers torch \
             scikit-learn nltk wikipedia-api datasets scipy huggingface_hub -q
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('✅ Dependencies ready')

In [ ]:
# Cell 0.4 — Import thư viện và set seed cố định
import os, sys, pickle, shutil, zipfile, json, warnings, random, time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample
from scipy import stats as scipy_stats
from collections import Counter
warnings.filterwarnings('ignore')

sys.path.insert(0, '/content')
sys.path.insert(0, '/content/src')

from data_loader import load_liar
from paraphrase_augmentor import synonym_substitute, combined_augment

def set_all_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
print('✅ All imports OK, seed=42 fixed')

In [ ]:
# Cell 0.5 — Định nghĩa các class và hàm dùng chung

class ImprovedMLP(nn.Module):
    """MLP cải tiến: BatchNorm + Residual connection + Xavier init."""
    def __init__(self, input_dim=384):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(0.4))
        self.block2 = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(0.3))
        self.block3 = nn.Sequential(
            nn.Linear(128, 64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(64, 2)
        self.residual = nn.Linear(input_dim, 128)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        res = self.residual(x)
        out = self.block1(x)
        out = self.block2(out) + res
        out = self.block3(out)
        return self.classifier(out)


def encode_pairs(claims, evidences, encoder):
    combined = [f'{c} [SEP] {e}' if e else c
                for c, e in zip(claims, evidences)]
    return encoder.encode(combined, batch_size=32, show_progress_bar=False)


def add_full_context(df):
    return [f"[{row['speaker']}] [{row['subject']}] {row['claim']}"
            for _, row in df.iterrows()]


def train_mlp(model, X_train, y_train, class_weights, device,
              epochs=25, lr=2e-4, seed=42, patience=6, label_smoothing=0.1):
    torch.manual_seed(seed)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    full_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    val_size = int(0.1 * len(full_ds))
    g = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(full_ds, [len(full_ds)-val_size, val_size], generator=g)
    loader_tr = DataLoader(train_ds, batch_size=64, shuffle=True)
    loader_val = DataLoader(val_ds, batch_size=64)

    total_steps = len(loader_tr) * epochs
    warmup_steps = max(1, total_steps // 10)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        return max(0.0, 1-(step-warmup_steps)/max(1, total_steps-warmup_steps))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val, patience_counter, step = float('inf'), 0, 0
    best_path = f'/content/best_mlp_{seed}_{id(model)}.pt'

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader_tr:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); step += 1

        model.eval()
        val_loss = sum(criterion(model(xb.to(device)), yb.to(device)).item()
                       for xb, yb in loader_val) / len(loader_val)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), best_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(torch.load(best_path, map_location=device))
    return model


def ensemble_predict(model, lr_model, X_emb, X_tfidf, device, w_arpd=0.10):
    model.eval()
    with torch.no_grad():
        prob_mlp = torch.softmax(model(torch.FloatTensor(X_emb).to(device)), dim=1).cpu().numpy()
    prob_lr = lr_model.predict_proba(X_tfidf)
    return (1-w_arpd)*prob_lr + w_arpd*prob_mlp


def evaluate(labels, preds):
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'f1_fake': f1_score(labels, preds, pos_label=0),
        'f1_real': f1_score(labels, preds, pos_label=1),
    }

print('✅ Classes và functions đã định nghĩa')

In [ ]:
# Cell 0.6 — Upload và giải nén LIAR dataset
from google.colab import files
import shutil, zipfile, os

os.makedirs('/content/data/raw', exist_ok=True)

drive_liar = '/content/drive/MyDrive/ARPD-research/liar_dataset.zip'
if os.path.exists(drive_liar):
    print('Tìm thấy liar_dataset.zip trên Drive, dùng luôn...')
    shutil.copy(drive_liar, '/content/data/raw/liar_dataset.zip')
else:
    print('Chưa có trên Drive — vui lòng upload liar_dataset.zip:')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.move(fname, '/content/data/raw/liar_dataset.zip')
    # Lưu lại lên Drive để lần sau không cần upload
    os.makedirs('/content/drive/MyDrive/ARPD-research', exist_ok=True)
    shutil.copy('/content/data/raw/liar_dataset.zip', drive_liar)

with zipfile.ZipFile('/content/data/raw/liar_dataset.zip', 'r') as z:
    z.extractall('/content/data/raw/')

print('✅ LIAR dataset ready')

In [ ]:
# Cell 0.7 — Load LIAR + Encode + TF-IDF
train_df = load_liar(split='train', cache_dir='/content/data/raw')
val_df   = load_liar(split='validation', cache_dir='/content/data/raw')
test_df  = load_liar(split='test', cache_dir='/content/data/raw')
print(f'✅ LIAR: train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}')

from sentence_transformers import SentenceTransformer
model_st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('✅ MiniLM loaded')

# Lấy cached evidences từ Drive nếu có, không thì để rỗng (retrieval bỏ qua)
ev_train_path = '/content/drive/MyDrive/ARPD-research/results/train_evidences.pkl'
ev_test_path  = '/content/drive/MyDrive/ARPD-research/results/test_evidences.pkl'

if os.path.exists(ev_train_path) and os.path.exists(ev_test_path):
    with open(ev_train_path, 'rb') as f:
        train_evidences = pickle.load(f)
    with open(ev_test_path, 'rb') as f:
        test_evidences = pickle.load(f)
    print(f'✅ Evidences đã cache: train={len(train_evidences):,}, test={len(test_evidences):,}')
else:
    print('⚠️  Không tìm thấy cached evidences — dùng claim-only (không retrieval)')
    train_evidences = [''] * len(train_df)
    test_evidences  = [''] * len(test_df)

print('\nEncoding LIAR claims...')
train_claims = add_full_context(train_df)
test_claims  = add_full_context(test_df)
X_train_emb = encode_pairs(train_claims, train_evidences, model_st)
X_test_emb  = encode_pairs(test_claims, test_evidences, model_st)
print(f'✅ Embeddings: train={X_train_emb.shape}, test={X_test_emb.shape}')

tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,3))
X_train_tfidf = tfidf.fit_transform(train_claims)
X_test_tfidf  = tfidf.transform(test_claims)

cw_vals = compute_class_weight('balanced', classes=np.array([0,1]), y=train_df['label'].values)
class_weights = torch.FloatTensor(cw_vals).to(device)
print(f'✅ Class weights: FAKE={cw_vals[0]:.3f}, REAL={cw_vals[1]:.3f}')
print('\n🚀 Setup hoàn tất!')

---
## Section 1 — Baselines trên LIAR

In [ ]:
# Cell 1.1 — Baseline: TF-IDF + Logistic Regression
lr_base = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
lr_base.fit(X_train_tfidf, train_df['label'])
res_lr = evaluate(test_df['label'], lr_base.predict(X_test_tfidf))
print('TF-IDF + LR:', res_lr)

In [ ]:
# Cell 1.2 — Baseline: TF-IDF + SVM
svm = CalibratedClassifierCV(LinearSVC(max_iter=2000, C=1.0))
svm.fit(X_train_tfidf, train_df['label'])
res_svm = evaluate(test_df['label'], svm.predict(X_test_tfidf))
print('TF-IDF + SVM:', res_svm)

In [ ]:
# Cell 1.3 — Baseline: MiniLM + SimpleMLP
class SimpleMLP(nn.Module):
    def __init__(self, input_dim=384):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 2))
    def forward(self, x): return self.net(x)

mlp_base = SimpleMLP().to(device)
mlp_base = train_mlp(mlp_base, X_train_emb, train_df['label'].tolist(),
                      class_weights, device, epochs=20, seed=42)
mlp_base.eval()
with torch.no_grad():
    preds_mlp = mlp_base(torch.FloatTensor(X_test_emb).to(device)).argmax(1).cpu().numpy()
res_mlp = evaluate(test_df['label'], preds_mlp)
print('MiniLM + MLP:', res_mlp)

---
## Section 2 — ARPD (ImprovedMLP + Ensemble) trên LIAR

In [ ]:
# Cell 2.1 — Train ImprovedMLP
set_all_seeds(42)
mlp_arpd = ImprovedMLP(input_dim=384).to(device)
print(f'Params: {sum(p.numel() for p in mlp_arpd.parameters()):,}')
mlp_arpd = train_mlp(mlp_arpd, X_train_emb, train_df['label'].tolist(),
                      class_weights, device, epochs=25, seed=42)
print('✅ ImprovedMLP trained')

In [ ]:
# Cell 2.2 — Grid search ensemble weight
best_acc, best_w, best_preds = 0, 0.10, None
for w in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35]:
    prob = ensemble_predict(mlp_arpd, lr_base, X_test_emb, X_test_tfidf, device, w)
    preds = prob.argmax(1)
    acc = accuracy_score(test_df['label'], preds)
    f1 = f1_score(test_df['label'], preds, average='macro')
    mark = ' <- best' if acc > best_acc else ''
    print(f'  w={w:.2f}: Acc={acc:.4f}, F1={f1:.4f}{mark}')
    if acc > best_acc:
        best_acc, best_w, best_preds = acc, w, preds

res_arpd = evaluate(test_df['label'], best_preds)
print(f'\n✅ Best weight: {best_w}')
print('ARPD:', res_arpd)
print(f"\n{classification_report(test_df['label'], best_preds, target_names=['FAKE','REAL'])}")

---
## Section 3 — Statistical Significance (5 seeds)

In [ ]:
# Cell 3.1 — Chạy 5 seeds
SEEDS = [42, 123, 456, 789, 2025]
sig_results = {'arpd': [], 'lr': []}

for i, seed in enumerate(SEEDS):
    print(f'--- Seed {seed} ({i+1}/{len(SEEDS)}) ---')
    set_all_seeds(seed)

    m = ImprovedMLP(input_dim=384).to(device)
    m = train_mlp(m, X_train_emb, train_df['label'].tolist(),
                  class_weights, device, epochs=25, seed=seed)

    best_acc_s, best_p_s = 0, None
    prob_lr_s = lr_base.predict_proba(X_test_tfidf)
    m.eval()
    with torch.no_grad():
        prob_m_s = torch.softmax(m(torch.FloatTensor(X_test_emb).to(device)), dim=1).cpu().numpy()
    for w in [0.10, 0.15, 0.20, 0.25, 0.30]:
        p = (1-w)*prob_lr_s + w*prob_m_s
        a = accuracy_score(test_df['label'], p.argmax(1))
        if a > best_acc_s:
            best_acc_s, best_p_s = a, p

    preds_s = best_p_s.argmax(1)
    sig_results['arpd'].append(evaluate(test_df['label'], preds_s))

    lr_s = LogisticRegression(max_iter=1000, C=0.5, random_state=seed)
    lr_s.fit(X_train_tfidf, train_df['label'])
    sig_results['lr'].append(evaluate(test_df['label'], lr_s.predict(X_test_tfidf)))

    print(f"  ARPD: Acc={sig_results['arpd'][-1]['accuracy']:.4f}, F1={sig_results['arpd'][-1]['f1_macro']:.4f}")
    print(f"  LR:   Acc={sig_results['lr'][-1]['accuracy']:.4f}, F1={sig_results['lr'][-1]['f1_macro']:.4f}")

In [ ]:
# Cell 3.2 — Tổng hợp và t-test
arpd_accs = [r['accuracy'] for r in sig_results['arpd']]
arpd_f1s  = [r['f1_macro'] for r in sig_results['arpd']]
arpd_ffs  = [r['f1_fake'] for r in sig_results['arpd']]
lr_accs   = [r['accuracy'] for r in sig_results['lr']]
lr_f1s    = [r['f1_macro'] for r in sig_results['lr']]
lr_ffs    = [r['f1_fake'] for r in sig_results['lr']]

t_acc, p_acc_2s = scipy_stats.ttest_rel(arpd_accs, lr_accs)
t_f1,  p_f1_2s  = scipy_stats.ttest_rel(arpd_f1s, lr_f1s)
t_ff,  p_ff_2s  = scipy_stats.ttest_rel(arpd_ffs, lr_ffs)

p_acc = p_acc_2s/2 if t_acc > 0 else 1.0
p_f1  = p_f1_2s/2  if t_f1  > 0 else 1.0
p_ff  = p_ff_2s/2  if t_ff  > 0 else 1.0

print('='*60)
print('STATISTICAL SIGNIFICANCE (one-sided paired t-test)')
print('='*60)
print(f'Accuracy: ARPD={np.mean(arpd_accs):.4f}±{np.std(arpd_accs):.4f} | LR={np.mean(lr_accs):.4f} | p={p_acc:.4f}')
print(f'F1-Macro: ARPD={np.mean(arpd_f1s):.4f}±{np.std(arpd_f1s):.4f} | LR={np.mean(lr_f1s):.4f} | p={p_f1:.4f} {"✅" if p_f1<0.05 else "⚠️"}')
print(f'F1-FAKE:  ARPD={np.mean(arpd_ffs):.4f}±{np.std(arpd_ffs):.4f} | LR={np.mean(lr_ffs):.4f} | p={p_ff:.4f} {"✅" if p_ff<0.05 else "⚠️"}')

---
## Section 4 — Robustness (Paraphrase Attack p=0.50)

In [ ]:
# Cell 4.1 — Robustness test trên 5 seeds
rob_arpd, rob_lr = [], []

for seed in SEEDS:
    set_all_seeds(seed)
    para_claims = [synonym_substitute(r['claim'], p=0.5, seed=seed) for _, r in test_df.iterrows()]
    para_ctx = [f"[{r['speaker']}] [{r['subject']}] {p}" for (_, r), p in zip(test_df.iterrows(), para_claims)]
    X_para_emb = encode_pairs(para_ctx, test_evidences, model_st)
    X_para_tfidf = tfidf.transform(para_ctx)

    m = ImprovedMLP(input_dim=384).to(device)
    ckpt = f'/content/best_mlp_{seed}_{id(mlp_arpd)}.pt'
    # fallback: retrain nhanh nếu checkpoint không khớp id
    m = train_mlp(m, X_train_emb, train_df['label'].tolist(), class_weights, device, epochs=25, seed=seed)

    prob_lr_c = lr_base.predict_proba(X_test_tfidf)
    with torch.no_grad():
        prob_m_c = torch.softmax(m(torch.FloatTensor(X_test_emb).to(device)), dim=1).cpu().numpy()
    acc_c = accuracy_score(test_df['label'], (0.90*prob_lr_c + 0.10*prob_m_c).argmax(1))

    prob_lr_p = lr_base.predict_proba(X_para_tfidf)
    with torch.no_grad():
        prob_m_p = torch.softmax(m(torch.FloatTensor(X_para_emb).to(device)), dim=1).cpu().numpy()
    acc_p = accuracy_score(test_df['label'], (0.90*prob_lr_p + 0.10*prob_m_p).argmax(1))
    rob_arpd.append(acc_c - acc_p)

    lr_s = LogisticRegression(max_iter=1000, C=0.5, random_state=seed)
    lr_s.fit(X_train_tfidf, train_df['label'])
    acc_lr_c = accuracy_score(test_df['label'], lr_s.predict(X_test_tfidf))
    acc_lr_p = accuracy_score(test_df['label'], lr_s.predict(X_para_tfidf))
    rob_lr.append(acc_lr_c - acc_lr_p)

    print(f'Seed {seed}: ARPD drop={rob_arpd[-1]:+.4f} | LR drop={rob_lr[-1]:+.4f}')

t_rob, p_rob_2s = scipy_stats.ttest_rel(rob_lr, rob_arpd)
p_rob = p_rob_2s/2 if t_rob > 0 else 1.0
print(f"\nTF-IDF+LR drop: {np.mean(rob_lr):.4f}±{np.std(rob_lr):.4f}")
print(f"ARPD drop:      {np.mean(rob_arpd):.4f}±{np.std(rob_arpd):.4f}")
print(f"p={p_rob:.4f} {'✅ ARPD more robust' if p_rob<0.05 else '⚠️ Not significant'}")

---
## Section 5 — Ablation Study

In [ ]:
# Cell 5.1 — Ablation: bỏ từng component
ablation = {'Full ARPD': res_arpd}

# Bỏ retrieval (claim only)
X_train_noret = model_st.encode(train_claims, batch_size=32, show_progress_bar=False)
X_test_noret  = model_st.encode(test_claims, batch_size=32, show_progress_bar=False)
m1 = ImprovedMLP(input_dim=384).to(device)
m1 = train_mlp(m1, X_train_noret, train_df['label'].tolist(), class_weights, device, epochs=20, seed=42)
m1.eval()
with torch.no_grad():
    prob1 = torch.softmax(m1(torch.FloatTensor(X_test_noret).to(device)), dim=1).cpu().numpy()
prob_lr_full = lr_base.predict_proba(X_test_tfidf)
ablation['No Retrieval'] = evaluate(test_df['label'], (0.90*prob_lr_full + 0.10*prob1).argmax(1))
print('✅ No Retrieval done')

# Bỏ speaker/subject context
train_claims_nos = train_df['claim'].tolist()
test_claims_nos  = test_df['claim'].tolist()
X_train_nos = encode_pairs(train_claims_nos, train_evidences, model_st)
X_test_nos  = encode_pairs(test_claims_nos, test_evidences, model_st)
tfidf_nos = TfidfVectorizer(max_features=15000, ngram_range=(1,3))
Xtr_tfidf_nos = tfidf_nos.fit_transform(train_claims_nos)
Xte_tfidf_nos = tfidf_nos.transform(test_claims_nos)
lr_nos = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
lr_nos.fit(Xtr_tfidf_nos, train_df['label'])
m2 = ImprovedMLP(input_dim=384).to(device)
m2 = train_mlp(m2, X_train_nos, train_df['label'].tolist(), class_weights, device, epochs=20, seed=42)
m2.eval()
with torch.no_grad():
    prob2 = torch.softmax(m2(torch.FloatTensor(X_test_nos).to(device)), dim=1).cpu().numpy()
prob_lr_nos = lr_nos.predict_proba(Xte_tfidf_nos)
ablation['No Speaker'] = evaluate(test_df['label'], (0.90*prob_lr_nos + 0.10*prob2).argmax(1))
print('✅ No Speaker Context done')

# Bỏ ensemble (MLP only)
mlp_arpd.eval()
with torch.no_grad():
    preds_noens = mlp_arpd(torch.FloatTensor(X_test_emb).to(device)).argmax(1).cpu().numpy()
ablation['No Ensemble'] = evaluate(test_df['label'], preds_noens)
print('✅ No Ensemble done')

print(f"\n{'Variant':<20} {'Accuracy':>10} {'F1-Macro':>10} {'Drop':>10}")
full_f1 = ablation['Full ARPD']['f1_macro']
for name, r in ablation.items():
    drop = f"{r['f1_macro']-full_f1:+.4f}" if name != 'Full ARPD' else '—'
    print(f"{name:<20} {r['accuracy']:>10.4f} {r['f1_macro']:>10.4f} {drop:>10}")

---
## Section 6 — FEVER Dataset + Wikipedia Corpus
> Phần này download train/dev jsonl của FEVER + toàn bộ Wikipedia corpus (wiki-pages.zip ~1.7GB) và build index để tra cứu evidence text thật. Mất khoảng **15-25 phút** tổng cộng tùy tốc độ mạng.

In [ ]:
# Cell 6.1 — Download FEVER train/dev + Wikipedia corpus
import urllib.request

FEVER_DIR = '/content/data/fever'
os.makedirs(FEVER_DIR, exist_ok=True)

urls = {
    'train': ('https://fever.ai/download/fever/train.jsonl', f'{FEVER_DIR}/train.jsonl'),
    'dev':   ('https://fever.ai/download/fever/shared_task_dev.jsonl', f'{FEVER_DIR}/dev.jsonl'),
}
for name, (url, path) in urls.items():
    if not os.path.exists(path):
        print(f'Downloading {name}...')
        urllib.request.urlretrieve(url, path)
    print(f'✅ {name}: {path}')

# Wikipedia corpus (~1.7GB) — phần "wiki" bạn cần
wiki_zip = f'{FEVER_DIR}/wiki-pages.zip'
wiki_dir = f'{FEVER_DIR}/wiki-pages'

if not os.path.exists(wiki_zip):
    print('\nDownloading wiki-pages.zip (~1.7GB, mất 5-10 phút)...')
    urllib.request.urlretrieve('https://fever.ai/download/fever/wiki-pages.zip', wiki_zip)
    print('✅ Downloaded')

if not os.path.exists(wiki_dir):
    print('Extracting...')
    with zipfile.ZipFile(wiki_zip, 'r') as z:
        z.extractall(FEVER_DIR)
    print('✅ Extracted')

print(f'\nSố file wiki-*.jsonl: {len(os.listdir(wiki_dir))}')

In [ ]:
# Cell 6.2 — Build wiki_index (đọc 5.4M trang Wikipedia, mất 5-10 phút)
drive_index_path = '/content/drive/MyDrive/ARPD-research/results/fever_wiki_index.pkl'

if os.path.exists(drive_index_path):
    print('Tìm thấy wiki_index đã build sẵn trên Drive, load luôn...')
    with open(drive_index_path, 'rb') as f:
        wiki_index = pickle.load(f)
    print(f'✅ Loaded: {len(wiki_index):,} trang')
else:
    print('Building wiki index từ đầu...')
    wiki_files = sorted(os.listdir(wiki_dir))
    wiki_index = {}

    for i, fname in enumerate(wiki_files):
        with open(f'{wiki_dir}/{fname}', 'r', encoding='utf-8') as f:
            for line in f:
                d = json.loads(line)
                title = d['id']
                sents = {}
                for sent_line in d.get('lines', '').split('\n'):
                    parts = sent_line.split('\t')
                    if len(parts) >= 2:
                        try:
                            sid = int(parts[0])
                            stext = parts[1].strip()
                            if stext:
                                sents[sid] = stext
                        except ValueError:
                            continue
                wiki_index[title] = sents
        if (i+1) % 10 == 0:
            print(f'  {i+1}/{len(wiki_files)} files, {len(wiki_index):,} trang')

    print(f'✅ Wiki index hoàn tất: {len(wiki_index):,} trang')

    # Lưu lên Drive để lần sau không phải build lại
    os.makedirs('/content/drive/MyDrive/ARPD-research/results', exist_ok=True)
    with open(drive_index_path, 'wb') as f:
        pickle.dump(wiki_index, f)
    print(f'✅ Đã lưu lên Drive: {drive_index_path}')

In [ ]:
# Cell 6.3 — Extract evidence text thật + binarize nhãn FEVER
def extract_evidence_text(evidence_groups, wiki_index, max_sentences=3):
    texts, seen = [], set()
    for group in evidence_groups:
        for ev in group:
            if len(ev) >= 4:
                page_title, sent_id = ev[2], ev[3]
                if page_title and sent_id is not None:
                    key = (page_title, sent_id)
                    if key in seen:
                        continue
                    seen.add(key)
                    text = wiki_index.get(page_title, {}).get(int(sent_id), '')
                    if text:
                        texts.append(text)
    return ' '.join(texts[:max_sentences])

def load_fever(path, wiki_index, max_samples=None):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            d = json.loads(line)
            if d['label'] == 'NOT ENOUGH INFO':
                continue
            ev_text = extract_evidence_text(d.get('evidence', []), wiki_index)
            data.append({
                'claim': d['claim'],
                'evidence': ev_text,
                'label': 1 if d['label'] == 'SUPPORTS' else 0,
                'label_str': d['label'],
            })
    return pd.DataFrame(data)

print('Loading FEVER train...')
fever_train_df = load_fever(f'{FEVER_DIR}/train.jsonl', wiki_index)
print(f'✅ Train: {len(fever_train_df):,}')

print('Loading FEVER dev...')
fever_test_df = load_fever(f'{FEVER_DIR}/dev.jsonl', wiki_index)
print(f'✅ Dev: {len(fever_test_df):,}')

cov_train = (fever_train_df['evidence'] != '').mean() * 100
cov_test  = (fever_test_df['evidence'] != '').mean() * 100
print(f'\nEvidence coverage — Train: {cov_train:.1f}% | Dev: {cov_test:.1f}%')

fever_train_df.to_csv('/content/drive/MyDrive/ARPD-research/fever_train_full.csv', index=False)
fever_test_df.to_csv('/content/drive/MyDrive/ARPD-research/fever_test_full.csv', index=False)
print('✅ Đã lưu CSV lên Drive')

---
## Section 7 — ARPD trên FEVER (Cross-domain)

In [ ]:
# Cell 7.1 — Balance + Encode FEVER
fever_real = fever_train_df[fever_train_df['label']==1]
fever_fake = fever_train_df[fever_train_df['label']==0]
fever_real_bal = resample(fever_real, n_samples=len(fever_fake), random_state=42)
fever_train_bal = pd.concat([fever_real_bal, fever_fake]).sample(frac=1, random_state=42)

fever_train_sub = fever_train_bal.head(10000).copy()
fever_test_sub  = fever_test_df.head(5000).copy()
print(f'Balanced train: {len(fever_train_sub):,} | Test: {len(fever_test_sub):,}')

def encode_fever(df, encoder):
    combined = [f"{r['claim']} [SEP] {r['evidence']}" if r['evidence'] else r['claim']
                for _, r in df.iterrows()]
    return encoder.encode(combined, batch_size=64, show_progress_bar=False)

print('Encoding FEVER...')
X_fever_train_emb = encode_fever(fever_train_sub, model_st)
X_fever_test_emb  = encode_fever(fever_test_sub, model_st)

tfidf_fever = TfidfVectorizer(max_features=15000, ngram_range=(1,3))
X_fever_tr_tfidf = tfidf_fever.fit_transform(fever_train_sub['claim'])
X_fever_te_tfidf = tfidf_fever.transform(fever_test_sub['claim'])

cw_fever_vals = compute_class_weight('balanced', classes=np.array([0,1]), y=fever_train_sub['label'].values)
cw_fever = torch.FloatTensor(cw_fever_vals).to(device)
print('✅ FEVER encoded')

In [ ]:
# Cell 7.2 — Train + Evaluate ARPD trên FEVER
lr_fever = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
lr_fever.fit(X_fever_tr_tfidf, fever_train_sub['label'])
res_lr_fever = evaluate(fever_test_sub['label'], lr_fever.predict(X_fever_te_tfidf))

mlp_fever = ImprovedMLP(input_dim=384).to(device)
mlp_fever = train_mlp(mlp_fever, X_fever_train_emb, fever_train_sub['label'].tolist(),
                      cw_fever, device, epochs=20, seed=42)

best_acc_f, best_preds_f = 0, None
prob_lr_f = lr_fever.predict_proba(X_fever_te_tfidf)
mlp_fever.eval()
with torch.no_grad():
    prob_m_f = torch.softmax(mlp_fever(torch.FloatTensor(X_fever_test_emb).to(device)), dim=1).cpu().numpy()
for w in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]:
    p = (1-w)*prob_lr_f + w*prob_m_f
    a = accuracy_score(fever_test_sub['label'], p.argmax(1))
    if a > best_acc_f:
        best_acc_f, best_preds_f = a, p.argmax(1)

res_arpd_fever = evaluate(fever_test_sub['label'], best_preds_f)

print(f"{'Model':<20} {'Accuracy':>10} {'F1-Macro':>10} {'F1-FAKE':>10}")
print(f"{'TF-IDF+LR':<20} {res_lr_fever['accuracy']:>10.4f} {res_lr_fever['f1_macro']:>10.4f} {res_lr_fever['f1_fake']:>10.4f}")
print(f"{'ARPD (FEVER)':<20} {res_arpd_fever['accuracy']:>10.4f} {res_arpd_fever['f1_macro']:>10.4f} {res_arpd_fever['f1_fake']:>10.4f}")

---
## Section 8 — Export kết quả cuối cùng

In [ ]:
# Cell 8.1 — Tổng hợp và lưu toàn bộ kết quả
from datetime import datetime

all_results = {
    'timestamp': datetime.now().isoformat(),
    'liar': {
        'baselines': {'tfidf_lr': res_lr, 'tfidf_svm': res_svm, 'minilm_mlp': res_mlp},
        'arpd': res_arpd,
        'statistical_significance': {
            'arpd_acc_mean': float(np.mean(arpd_accs)), 'arpd_acc_std': float(np.std(arpd_accs)),
            'arpd_f1_mean': float(np.mean(arpd_f1s)), 'arpd_f1_std': float(np.std(arpd_f1s)),
            'arpd_ff_mean': float(np.mean(arpd_ffs)), 'arpd_ff_std': float(np.std(arpd_ffs)),
            'lr_acc_mean': float(np.mean(lr_accs)), 'lr_f1_mean': float(np.mean(lr_f1s)),
            'p_value_acc': float(p_acc), 'p_value_f1': float(p_f1), 'p_value_f1fake': float(p_ff),
        },
        'robustness': {
            'arpd_drop_mean': float(np.mean(rob_arpd)), 'arpd_drop_std': float(np.std(rob_arpd)),
            'lr_drop_mean': float(np.mean(rob_lr)), 'lr_drop_std': float(np.std(rob_lr)),
            'p_value': float(p_rob),
        },
        'ablation': ablation,
    },
    'fever': {
        'evidence_coverage_train': float(cov_train),
        'evidence_coverage_dev': float(cov_test),
        'tfidf_lr': res_lr_fever,
        'arpd': res_arpd_fever,
    },
}

out_path = '/content/drive/MyDrive/ARPD-research/results/FINAL_RESULTS_FULL.json'
with open(out_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'✅ Đã lưu: {out_path}')

print(f"\n{'='*70}")
print('BẢNG TỔNG KẾT CUỐI CÙNG')
print(f"{'='*70}")
print(f"{'Model':<22} {'Dataset':>8} {'Acc':>8} {'F1':>8} {'F1-FAKE':>8}")
rows = [
    ('TF-IDF+LR', 'LIAR', res_lr), ('TF-IDF+SVM', 'LIAR', res_svm),
    ('MiniLM+MLP', 'LIAR', res_mlp), ('ARPD', 'LIAR', res_arpd),
    ('TF-IDF+LR', 'FEVER', res_lr_fever), ('ARPD', 'FEVER', res_arpd_fever),
]
for name, ds, r in rows:
    print(f"{name:<22} {ds:>8} {r['accuracy']:>8.4f} {r['f1_macro']:>8.4f} {r['f1_fake']:>8.4f}")
print('\n🎉 Toàn bộ pipeline LIAR + FEVER + Wikipedia hoàn tất!')